In [0]:
%pip install --upgrade databricks-sdk>=0.68.0 psycopg pyyaml
%restart_python

In [0]:
%run ./0-Parameters

## Parameters
This notebook contains the parameters needed to customise the solution accelerator to you environment. Be sure to modify them before starting to ensure that the accelerator deploys correctly.

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors.platform import NotFound
from databricks.sdk.service.apps import App, AppResource, AppResourceDatabase, AppResourceDatabaseDatabasePermission, \
    AppResourceSqlWarehouse, AppResourceSqlWarehouseSqlWarehousePermission
from databricks.sdk.service.catalog import PermissionsChange, Privilege
import psycopg
import yaml
import uuid

w = WorkspaceClient()

### Preparing the app 
In the cells below:
1. We dynamically create the app.yaml file 
2. We copy the contents of the deployment-staging folder into a deployment folder (limitation of apps integration with GIT repo's)

In [ ]:
#we define the schema where the table with RDF definitions will be stored. 
DIGITAL_TWIN_SCHEMA = "digitaltwin"

In [0]:
# We generate the app.yaml payload from the parameters notebook.
# The file will be written into the final deploy folder (after it is created).
app_yaml = {
    "command": ["python", "server.py"],
    "env": [
        {"name": "WAREHOUSE_ID", "valueFrom": "sql_warehouse"},
        {"name": "SYNCED_TABLE_FULL_NAME", "value": SYNCED_TABLE_FULL_NAME_PG},
        {"name": "TRIPLE_TABLE_FULL_NAME", "value": TRIPLE_TABLE_FULL_NAME},
        {"name": "DATABRICKS_CATALOG", "value": TRIPLE_CATALOG},
        {"name": "DATABRICKS_SCHEMA", "value": TRIPLE_SCHEMA},
        {"name": "DATABRICKS_TABLE", "value": TRIPLE_TABLE},
        {"name": "LAKEBASE_INSTANCE_NAME", "value": INSTANCE_NAME},
        {"name": "PG_TOKEN_REFRESH_SECONDS", "value": 900},
        {"name": "RDF_MODELS_FULL_TABLE_NAME", "value": DIGITAL_TWIN_SCHEMA + "." + DIGITAL_TWIN_TABLE},
        {"name": "PGDATABASE", "value": INSTANCE_NAME},
    ],
}


In [ ]:
import shutil
import os

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
parent_path = os.path.dirname(notebook_path)
grandparent_path = os.path.dirname(parent_path)

# /Workspace/Users/<user>/.../deployment-staging
src_folder = "/Workspace" + parent_path + "/deployment-staging/"
# /Workspace/Users/<user>/.../deployment-digital-twin
dst_folder = "/Workspace" + grandparent_path + "/deployment-digital-twin"

if os.path.exists(dst_folder):
    shutil.rmtree(dst_folder)

shutil.copytree(src_folder, dst_folder)

# Write app.yaml into the *deploy* folder so the deployment always includes it
with open(os.path.join(dst_folder, "app.yaml"), "w") as f:
    yaml.dump(app_yaml, f)

# Apps deployments expect a workspace path (do not use local abspath())
APP_SRC_CODE_PATH = dst_folder


### Setting up the app 
In the cells below we:
1. Create an app and attach the required resources, which are a SQL warehouse and Lakebase instance. 
2. We inject the proper URL into the pre-packaged app bundle 
3. We then generate a new deployment for the app 


In [0]:

app_conf = App(
    name=APP_NAME,
    default_source_code_path=APP_SRC_CODE_PATH,
    description="SPARQL query frontend",
    # effective_user_api_scopes=["iam.current-user:read", "iam.access-control:read"],
    resources=[
        AppResource(
            name="database",
            database=AppResourceDatabase(
                instance_name=INSTANCE_NAME,
                database_name=INSTANCE_NAME,
                permission=AppResourceDatabaseDatabasePermission.CAN_CONNECT_AND_CREATE,
            ),
            description="Low latency serving of latest triples.",
        ),
        AppResource(
            name="sql_warehouse",
            sql_warehouse=AppResourceSqlWarehouse(
                id=WAREHOUSE_ID,
                permission=AppResourceSqlWarehouseSqlWarehousePermission.CAN_USE,
            ),
            description="Serving of all historical data.",
        ),
    ],
)

try:
    app = w.apps.get(APP_NAME)
    print(f"App {app.name} already exists")
except NotFound:
    app = w.apps.create_and_wait(app_conf)
    app = w.apps.get(APP_NAME)
    print(f"Created app {app.name}")



In [ ]:
#Replace the templated app url with the generated app url 
import os

js_dir = dst_folder+"/dist"
js_files = []
for root, _, files in os.walk(js_dir):
    js_files.extend([os.path.join(root, f) for f in files if f.endswith('.js')])

for js_path in js_files:
    with open(js_path, "r", encoding="utf-8") as file:
        content = file.read()
    # Replace the specific URL with app.url
    updated_content = content.replace("BACKEND_URL_PLACEHOLDER", app.url)
    with open(js_path, "w", encoding="utf-8") as file:
        file.write(updated_content)

In [ ]:
from databricks.sdk.service.apps import AppDeployment, AppDeploymentMode

# Create a new AppDeployment instance.
# SNAPSHOT is the most reliable mode when deploying from a workspace folder.
new_deployment = AppDeployment(
    deployment_id=str(uuid.uuid4()),
    mode=AppDeploymentMode.SNAPSHOT,
    source_code_path=APP_SRC_CODE_PATH,
)

# Deploy and wait so failures are visible in the job output
try:
    deployment = w.apps.deploy_and_wait(APP_NAME, new_deployment)
    print(f"App {APP_NAME} deployment SUCCEEDED: {deployment.deployment_id}")
except Exception as e:
    print(f"App {APP_NAME} deployment FAILED: {e}")
    raise

# Ensure the app is started
try:
    w.apps.start(APP_NAME)
    print(f"App {APP_NAME} start requested")
except Exception as e:
    print(f"Warning: could not start app {APP_NAME}: {e}")


### Granting the right priviliges 

We make sure the Service principal of the app is able to access the necessary data 

In [0]:
r1 = w.grants.update(
    "TABLE",
    SYNCED_TABLE_FULL_NAME_UC,
    changes=[
        PermissionsChange(
            principal=app.service_principal_client_id,
            add=[Privilege.SELECT],
        )
    ],
)

r2 = w.grants.update(
    "TABLE",
    TRIPLE_TABLE_FULL_NAME,
    changes=[
        PermissionsChange(
            principal=app.service_principal_client_id,
            add=[Privilege.SELECT],
        )
    ],
)

In [0]:
instance = w.database.get_database_instance(INSTANCE_NAME)
cred = w.database.generate_database_credential(
    request_id=str(uuid.uuid4()), instance_names=[instance.name]
)
current_user = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook()
    .getContext()
    .userName()
    .getOrElse(None)
)

conn_conf = {
    "host": instance.read_write_dns,
    "port": 5432,
    "dbname": INSTANCE_NAME,
    "user": current_user,
    "password": cred.token,
    "sslmode": "require",
    "autocommit": True,
}

grants = f"""
GRANT USAGE ON SCHEMA {SYNCED_TABLE_SCHEMA} TO "{app.service_principal_client_id}";
GRANT SELECT ON TABLE {SYNCED_TABLE_SCHEMA}.{SYNCED_TABLE_NAME} TO "{app.service_principal_client_id}";
"""

with psycopg.connect(**conn_conf) as conn:
    with conn.cursor() as cur:
        cur.execute(grants)

### Creating the RDF models table
To allow the app to store digital twins, we create a table to store the underlying RDF models and
grant the appropriate priviliges to the service principal of the app. 

In [0]:
DDL_CREATE_TABLE = f"""
CREATE SCHEMA IF NOT EXISTS {DIGITAL_TWIN_SCHEMA}; 
 CREATE TABLE IF NOT EXISTS {DIGITAL_TWIN_SCHEMA}.{DIGITAL_TWIN_TABLE} (
                id SERIAL PRIMARY KEY,
                name VARCHAR(255) NOT NULL UNIQUE,
                description TEXT,
                category VARCHAR(50) DEFAULT 'user',
                is_template BOOLEAN DEFAULT FALSE,
                content TEXT NOT NULL,
                creator VARCHAR(255),
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                metadata JSONB DEFAULT '{{}}',
                tags TEXT[] DEFAULT ARRAY[]::TEXT[]
            );

            -- Create indexes for better performance
            CREATE INDEX IF NOT EXISTS idx_rdf_models_category ON {DIGITAL_TWIN_SCHEMA}.{DIGITAL_TWIN_TABLE} (category);
            CREATE INDEX IF NOT EXISTS idx_rdf_models_is_template ON {DIGITAL_TWIN_SCHEMA}.{DIGITAL_TWIN_TABLE} (is_template);
            CREATE INDEX IF NOT EXISTS idx_rdf_models_creator ON {DIGITAL_TWIN_SCHEMA}.{DIGITAL_TWIN_TABLE} (creator);
            CREATE INDEX IF NOT EXISTS idx_rdf_models_created_at ON {DIGITAL_TWIN_SCHEMA}.{DIGITAL_TWIN_TABLE} (created_at DESC);

GRANT USAGE, SELECT ON ALL SEQUENCES IN SCHEMA {DIGITAL_TWIN_SCHEMA} TO "{app.service_principal_client_id}";
GRANT USAGE ON SCHEMA {DIGITAL_TWIN_SCHEMA} TO "{app.service_principal_client_id}";
GRANT ALL PRIVILEGES ON TABLE {DIGITAL_TWIN_SCHEMA}.{DIGITAL_TWIN_TABLE} TO "{app.service_principal_client_id}";
"""

with psycopg.connect(**conn_conf) as conn:
    with conn.cursor() as cur:
        cur.execute(DDL_CREATE_TABLE)